# AU-FISH Multi-Seed Individual Run

**Protocol:** `time_based`  
**Model:** `YOLO11m`  
**Seed:** `123`  
**Experiment tag:** `multiseed_fullset_dedup_v1`

Notebook ini hanya menangani satu training run. Semua output disimpan pada folder
protocol, model, dan seed yang spesifik sehingga tidak bercampur dengan run lain.

Split dataset tidak diubah antar-seed. Preprocessing final tetap menggunakan:

- explicit image lists;
- filtered two-class annotations: coral and fish;
- exact duplicate removal yang telah dilakukan sebelumnya;
- minimum box width/height 5 px;
- minimum box area 25 px²;
- common filtered COCO ground truth.

Jangan mengubah protocol, model, seed, split, atau output root di tengah eksekusi.

In [ ]:
# CELL 1 — INSTALL PINNED DEPENDENCIES
!pip -q install ultralytics==8.4.102 pycocotools pandas pyyaml pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.3 MB/s eta 0:00:00


In [ ]:
# CELL 2 — MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# CELL 3 — FIXED RUN CONFIGURATION
import contextlib
import hashlib
import io
import json
import os
import platform
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from ultralytics import YOLO
MODEL_CLASS = YOLO

PROTOCOL = "time_based"
MODEL_KEY = "yolo11m"
MODEL_LABEL = "YOLO11m"
SEED = 123

PROJECT_ROOT = Path("/content/drive/MyDrive/au-fish/dataset_v2")
DATASET_ROOT = PROJECT_ROOT / PROTOCOL

INDIVIDUAL_TAG = "multiseed_fullset_dedup_v1"
LEGACY_FINAL_TAG = "simple_wbf_fullset_dedup_v1"

YAML_PATH = DATASET_ROOT / f"aufish_{PROTOCOL}_fullset.yaml"

TRAIN_RUN_ROOT = (
    DATASET_ROOT
    / "runs"
    / INDIVIDUAL_TAG
    / MODEL_KEY
    / f"seed_{SEED}"
)

EVAL_RUN_ROOT = (
    DATASET_ROOT
    / "runs_eval"
    / INDIVIDUAL_TAG
    / MODEL_KEY
    / f"seed_{SEED}"
)

EXPERIMENT_ROOT = (
    DATASET_ROOT
    / "experiment_logs"
    / INDIVIDUAL_TAG
)

RUN_ROOT = (
    EXPERIMENT_ROOT
    / "individual"
    / MODEL_KEY
    / f"seed_{SEED}"
)

RAW_DIR = RUN_ROOT / "raw_predictions"
RESULTS_DIR = RUN_ROOT / "results"
MANIFEST_DIR = RUN_ROOT / "manifests"
SHARED_GT_DIR = EXPERIMENT_ROOT / "shared_ground_truth"

for path in [
    TRAIN_RUN_ROOT,
    EVAL_RUN_ROOT,
    RAW_DIR,
    RESULTS_DIR,
    MANIFEST_DIR,
    SHARED_GT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

PRETRAINED_WEIGHTS = "yolo11m.pt"
EPOCHS = 100
IMAGE_SIZE = 640
TRAIN_BATCH = 16
PREDICT_BATCH = 8
DEVICE = 0 if torch.cuda.is_available() else "cpu"

TRAIN_RUN_NAME = "train"
TRAIN_OUTPUT_DIR = TRAIN_RUN_ROOT / TRAIN_RUN_NAME
BEST_CHECKPOINT = TRAIN_OUTPUT_DIR / "weights" / "best.pt"

FORCE_RETRAIN = False
FORCE_REINFERENCE = False

EXPECTED_IMAGES = {
    "time_based": {"train": 1061, "val": 203, "test": 282},
    "source_based": {"train": 1083, "val": 232, "test": 231},
}

print("=" * 80)
print("AU-FISH MULTI-SEED INDIVIDUAL RUN")
print("Protocol :", PROTOCOL)
print("Model    :", MODEL_LABEL)
print("Seed     :", SEED)
print("Run root :", RUN_ROOT)
print("GPU      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("=" * 80)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
AU-FISH MULTI-SEED INDIVIDUAL RUN
Protocol : time_based
Model    : YOLO11m
Seed     : 123
Run root : /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_fullset_dedup_v1/individual/yolo11m/seed_123
GPU      : NVIDIA L4


In [ ]:
# CELL 4 — REPRODUCIBILITY AND FILE HELPERS
def set_global_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(path.suffix + ".tmp")

    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary.replace(path)


def save_run_config():
    config = {
        "protocol": PROTOCOL,
        "model": MODEL_KEY,
        "seed": SEED,
        "experiment_tag": INDIVIDUAL_TAG,
        "dataset_yaml": str(YAML_PATH),
        "pretrained_weights": PRETRAINED_WEIGHTS,
        "epochs": EPOCHS,
        "image_size": IMAGE_SIZE,
        "train_batch": TRAIN_BATCH,
        "predict_batch": PREDICT_BATCH,
        "deterministic": True,
        "raw_prediction_confidence": 0.001,
        "raw_prediction_iou": 0.70,
        "raw_prediction_max_det": 300,
        "test_time_augmentation": False,
        "checkpoint_selection": "Ultralytics validation fitness",
    }

    save_json(
        config,
        MANIFEST_DIR / "run_config.json",
    )


set_global_seed(SEED)
save_run_config()
print("Global seed set:", SEED)

Global seed set: 123


In [ ]:
# CELL 5 — DATASET AND SHARED GROUND-TRUTH PREFLIGHT
if not YAML_PATH.exists():
    raise FileNotFoundError(
        f"Final explicit-list YAML not found: {YAML_PATH}"
    )

with open(YAML_PATH, "r", encoding="utf-8") as file:
    data_yaml = yaml.safe_load(file)

required_yaml_keys = {"train", "val", "test", "names"}

if not required_yaml_keys.issubset(data_yaml):
    raise RuntimeError(
        f"YAML missing keys: {required_yaml_keys - set(data_yaml)}"
    )

for split_name in ["train", "val", "test"]:
    list_path = Path(data_yaml[split_name])

    if not list_path.exists():
        raise FileNotFoundError(
            f"Explicit image list not found: {list_path}"
        )

    image_paths = [
        line.strip()
        for line in list_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    expected = EXPECTED_IMAGES[PROTOCOL][split_name]

    if len(image_paths) != expected:
        raise RuntimeError(
            f"{split_name}: expected {expected} images, "
            f"found {len(image_paths)}"
        )

    if len(image_paths) != len(set(image_paths)):
        raise RuntimeError(
            f"{split_name}: duplicate paths found"
        )

    missing = [
        path
        for path in image_paths
        if not Path(path).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"{split_name}: {len(missing)} image files missing. "
            f"First: {missing[0]}"
        )

    print(
        f"✅ {split_name}: {len(image_paths)} explicit paths"
    )

legacy_gt_dir = (
    DATASET_ROOT
    / "experiment_logs"
    / LEGACY_FINAL_TAG
    / "shared_ground_truth"
)

gt_files = {
    "validation": "val_gt_filtered.json",
    "test": "test_gt_filtered.json",
}

for split_label, file_name in gt_files.items():
    source_path = legacy_gt_dir / file_name
    target_path = SHARED_GT_DIR / file_name

    if not source_path.exists():
        raise FileNotFoundError(
            f"Locked common GT not found: {source_path}"
        )

    if not target_path.exists():
        shutil.copy2(source_path, target_path)

    gt = load_json(target_path)
    expected = EXPECTED_IMAGES[PROTOCOL][
        "val" if split_label == "validation" else "test"
    ]

    if len(gt["images"]) != expected:
        raise RuntimeError(
            f"{split_label} GT image count mismatch: "
            f"{len(gt['images'])} != {expected}"
        )

    print(
        f"✅ {split_label} GT: "
        f"{len(gt['images'])} images, "
        f"{len(gt['annotations'])} annotations"
    )

✅ train: 1061 explicit paths
✅ val: 203 explicit paths
✅ test: 282 explicit paths
✅ validation GT: 203 images, 913 annotations
✅ test GT: 282 images, 1400 annotations


In [ ]:
# CELL 6 — TRAIN YOLO11M
if BEST_CHECKPOINT.exists() and not FORCE_RETRAIN:
    print("Checkpoint already exists. Training skipped:")
    print(BEST_CHECKPOINT)
else:
    if TRAIN_OUTPUT_DIR.exists() and FORCE_RETRAIN:
        shutil.rmtree(TRAIN_OUTPUT_DIR)

    model = MODEL_CLASS(PRETRAINED_WEIGHTS)

    train_result = model.train(
        data=str(YAML_PATH),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=TRAIN_BATCH,
        device=DEVICE,
        seed=SEED,
        deterministic=True,
        project=str(TRAIN_RUN_ROOT),
        name=TRAIN_RUN_NAME,
        exist_ok=False,
        resume=False,
        verbose=True,
    )

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint not produced: {BEST_CHECKPOINT}"
    )

print("✅ Best checkpoint:", BEST_CHECKPOINT)
print("SHA256:", sha256_file(BEST_CHECKPOINT))

New https://pypi.org/project/ultralytics/8.4.107 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/au-fish/dataset_v2/time_based/aufish_time_based_fullset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

In [ ]:
# CELL 7 — FRAMEWORK-NATIVE VALIDATION AND TEST DIAGNOSTICS
diagnostic_rows = []

trained_model = MODEL_CLASS(str(BEST_CHECKPOINT))

for split_name in ["val", "test"]:
    output_name = f"{split_name}_diagnostic"

    metrics = trained_model.val(
        data=str(YAML_PATH),
        split=split_name,
        imgsz=IMAGE_SIZE,
        batch=PREDICT_BATCH,
        device=DEVICE,
        project=str(EVAL_RUN_ROOT),
        name=output_name,
        exist_ok=True,
        verbose=True,
    )

    row = {
        "protocol": PROTOCOL,
        "model": MODEL_KEY,
        "seed": SEED,
        "split": split_name,
        "mAP50_95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),
        "AP_coral": float(metrics.box.maps[0]),
        "AP_fish": float(metrics.box.maps[1]),
    }

    diagnostic_rows.append(row)

diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df.to_csv(
    RESULTS_DIR / "framework_native_results.csv",
    index=False,
)

display(diagnostic_df)

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11m summary (fused): 126 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.0 ms, read: 581.6±199.8 MB/s, size: 2614.9 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/time_based/val/labels.cache... 203 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 203/203 56.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 5.2it/s 5.0s
                   all        203        913       0.79      0.743      0.794      0.521
                 coral        126        280      0.801      0.804      0.848      0.632
                  fish        118        633       0.78      0.681      0.741       0.41
Speed: 0.6ms preprocess, 8.0ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to /content/drive/.shortcut-targets-by-

,protocol,model,seed,split,mAP50_95,mAP50,mAP75,AP_coral,AP_fish
0,time_based,yolo11m,123,val,0.521253,0.794316,0.542764,0.632069,0.410438
1,time_based,yolo11m,123,test,0.462741,0.718918,0.478227,0.537174,0.388309


In [11]:
# ============================================================
# CLEAN GPU MEMORY BEFORE RT-DETR TEST INFERENCE
# ============================================================
import gc
import torch

objects_to_delete = [
    "model",
    "trained_model",
    "inference_model",
    "train_result",
    "metrics",
    "results",
    "result",
]

for object_name in objects_to_delete:
    if object_name in globals():
        del globals()[object_name]
        print(f"Deleted: {object_name}")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    free, total = torch.cuda.mem_get_info()

    print(f"Allocated : {allocated:.2f} GiB")
    print(f"Reserved  : {reserved:.2f} GiB")
    print(f"Free GPU  : {free / 1024**3:.2f} GiB")
    print(f"Total GPU : {total / 1024**3:.2f} GiB")

Deleted: model
Deleted: trained_model
Deleted: inference_model
Deleted: train_result
Deleted: metrics
Allocated : 1.43 GiB
Reserved  : 8.83 GiB
Free GPU  : 12.89 GiB
Total GPU : 22.03 GiB


In [12]:
# CELL 8 — RAW PREDICTION AND COMMON COCOEVAL HELPERS
def blank_record(image_info):
    return {
        "image_id": int(image_info["id"]),
        "file_name": str(image_info["file_name"]),
        "width": int(image_info["width"]),
        "height": int(image_info["height"]),
        "boxes": [],
        "scores": [],
        "labels": [],
    }


def gt_path_for(split_label):
    file_name = (
        "val_gt_filtered.json"
        if split_label == "validation"
        else "test_gt_filtered.json"
    )

    return SHARED_GT_DIR / file_name


def split_dir_name(split_label):
    return "val" if split_label == "validation" else "test"


def build_ordered_image_paths(gt):
    split_dir = split_dir_name(CURRENT_SPLIT)

    paths = []

    for image_info in gt["images"]:
        path = (
            DATASET_ROOT
            / split_dir
            / "images"
            / image_info["file_name"]
        )

        if not path.exists():
            raise FileNotFoundError(
                f"Image missing: {path}"
            )

        paths.append(str(path))

    return paths


# ============================================================
# MEMORY-SAFE RT-DETR RAW INFERENCE
# Single-image streaming + partial resume
# ============================================================
import gc
from pathlib import Path


def run_raw_inference(model, split_label):
    """
    Memory-safe raw prediction.

    Fitur:
    - menggunakan satu gambar per inference;
    - tidak mengumpulkan seluruh Results di memori;
    - menyimpan progress sementara;
    - dapat dilanjutkan setelah runtime terputus;
    - mapping menggunakan urutan COCO ground truth;
    - validation/test final yang sudah lengkap akan dipakai ulang.
    """
    gt_path = gt_path_for(split_label)
    gt = load_json(gt_path)

    image_infos = gt["images"]

    split_directory = (
        "val"
        if split_label == "validation"
        else "test"
    )

    output_path = (
        RAW_DIR
        / f"{split_label}_{MODEL_KEY}.json"
    )

    partial_path = (
        RAW_DIR
        / f"{split_label}_{MODEL_KEY}.partial.json"
    )

    expected_images = len(image_infos)

    # Gunakan ulang output final jika sudah lengkap.
    if output_path.exists() and not FORCE_REINFERENCE:
        raw = load_json(output_path)

        if len(raw) != expected_images:
            raise RuntimeError(
                f"Existing {split_label} prediction is incomplete: "
                f"{len(raw)} != {expected_images}"
            )

        expected_ids = {
            str(int(image["id"]))
            for image in image_infos
        }

        actual_ids = set(raw.keys())

        if actual_ids != expected_ids:
            raise RuntimeError(
                f"{split_label} image-ID coverage mismatch. "
                f"Missing={len(expected_ids - actual_ids)}, "
                f"extra={len(actual_ids - expected_ids)}"
            )

        print(
            f"✅ Raw prediction already complete: "
            f"{output_path}"
        )

        return raw

    # Lanjutkan dari file partial apabila tersedia.
    if partial_path.exists():
        raw = load_json(partial_path)

        print(
            f"♻️ Resuming {split_label} inference: "
            f"{len(raw)}/{expected_images} images completed"
        )
    else:
        raw = {}

    model.predictor = None

    for position, image_info in enumerate(
        image_infos,
        start=1,
    ):
        image_id = str(int(image_info["id"]))

        # Lewati gambar yang telah tersimpan di partial file.
        if image_id in raw:
            continue

        image_path = (
            DATASET_ROOT
            / split_directory
            / "images"
            / image_info["file_name"]
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        record = blank_record(image_info)

        result_count = 0

        # stream=True menghindari materialisasi seluruh Results.
        result_stream = model.predict(
            source=[str(image_path)],
            conf=0.001,
            iou=0.70,
            max_det=300,
            imgsz=IMAGE_SIZE,
            batch=1,
            device=DEVICE,
            augment=False,
            save=False,
            stream=True,
            verbose=False,
        )

        for result in result_stream:
            result_count += 1

            if result_count > 1:
                raise RuntimeError(
                    f"More than one result returned for: "
                    f"{image_path}"
                )

            if result.boxes is not None:
                boxes = (
                    result.boxes.xyxy
                    .detach()
                    .cpu()
                    .numpy()
                )

                scores = (
                    result.boxes.conf
                    .detach()
                    .cpu()
                    .numpy()
                )

                labels = (
                    result.boxes.cls
                    .detach()
                    .cpu()
                    .numpy()
                )

                for box, score, label in zip(
                    boxes,
                    scores,
                    labels,
                ):
                    x1, y1, x2, y2 = map(
                        float,
                        box,
                    )

                    if x2 <= x1 or y2 <= y1:
                        continue

                    record["boxes"].append([
                        x1,
                        y1,
                        x2,
                        y2,
                    ])

                    record["scores"].append(
                        float(score)
                    )

                    record["labels"].append(
                        int(label)
                    )

            del result

        if result_count != 1:
            raise RuntimeError(
                f"No prediction result returned for: "
                f"{image_path}"
            )

        raw[image_id] = record

        # Simpan progress secara berkala.
        if (
            position % 10 == 0
            or position == expected_images
        ):
            save_json(
                raw,
                partial_path,
            )

            print(
                f"{split_label}: "
                f"{len(raw)}/{expected_images} images saved"
            )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    expected_ids = {
        str(int(image["id"]))
        for image in image_infos
    }

    actual_ids = set(raw.keys())

    if actual_ids != expected_ids:
        raise RuntimeError(
            f"Final {split_label} coverage mismatch. "
            f"Missing={len(expected_ids - actual_ids)}, "
            f"extra={len(actual_ids - expected_ids)}"
        )

    save_json(
        raw,
        output_path,
    )

    if partial_path.exists():
        partial_path.unlink()

    print(
        f"✅ {split_label} raw predictions completed: "
        f"{len(raw)} images"
    )

    print(f"Saved to: {output_path}")

    return raw


def raw_to_coco_rows(raw):
    rows = []

    for record in raw.values():
        for box, score, label in zip(
            record["boxes"],
            record["scores"],
            record["labels"],
        ):
            x1, y1, x2, y2 = map(float, box)
            width = x2 - x1
            height = y2 - y1

            if width <= 0 or height <= 0:
                continue

            rows.append({
                "image_id": int(record["image_id"]),
                "category_id": int(label) + 1,
                "bbox": [x1, y1, width, height],
                "score": float(score),
            })

    return rows


def evaluate_common_coco(gt_path, raw):
    rows = raw_to_coco_rows(raw)

    if not rows:
        raise RuntimeError("No predictions for COCOeval.")

    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(str(gt_path))
        coco_dt = coco_gt.loadRes(rows)

        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            "bbox",
        )
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    class_ap = {}

    for category_id, class_name in [
        (1, "coral"),
        (2, "fish"),
    ]:
        with contextlib.redirect_stdout(io.StringIO()):
            class_eval = COCOeval(
                coco_gt,
                coco_dt,
                "bbox",
            )
            class_eval.params.catIds = [
                category_id
            ]
            class_eval.evaluate()
            class_eval.accumulate()
            class_eval.summarize()

        class_ap[class_name] = float(
            class_eval.stats[0]
        )

    return {
        "mAP50_95": float(evaluator.stats[0]),
        "mAP50": float(evaluator.stats[1]),
        "mAP75": float(evaluator.stats[2]),
        "AP_coral": class_ap["coral"],
        "AP_fish": class_ap["fish"],
        "worst_class_AP": min(
            class_ap["coral"],
            class_ap["fish"],
        ),
        "detections": len(rows),
    }

In [13]:
# CELL 9 — GENERATE VALIDATION AND TEST RAW PREDICTIONS
inference_model = MODEL_CLASS(
    str(BEST_CHECKPOINT)
)

common_rows = []

for split_label in [
    "validation",
    "test",
]:
    raw = run_raw_inference(
        inference_model,
        split_label,
    )

    metrics = evaluate_common_coco(
        gt_path_for(split_label),
        raw,
    )

    common_rows.append({
        "protocol": PROTOCOL,
        "model": MODEL_KEY,
        "seed": SEED,
        "split": split_label,
        **metrics,
    })

common_df = pd.DataFrame(common_rows)
common_df.to_csv(
    RESULTS_DIR / "common_cocoeval_results.csv",
    index=False,
)

display(common_df)

✅ Raw prediction already complete: /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_fullset_dedup_v1/individual/yolo11m/seed_123/raw_predictions/validation_yolo11m.json
test: 10/282 images saved
test: 20/282 images saved
test: 30/282 images saved
test: 40/282 images saved
test: 50/282 images saved
test: 60/282 images saved
test: 70/282 images saved
test: 80/282 images saved
test: 90/282 images saved
test: 100/282 images saved
test: 110/282 images saved
test: 120/282 images saved
test: 130/282 images saved
test: 140/282 images saved
test: 150/282 images saved
test: 160/282 images saved
test: 170/282 images saved
test: 180/282 images saved
test: 190/282 images saved
test: 200/282 images saved
test: 210/282 images saved
test: 220/282 images saved
test: 230/282 images saved
test: 240/282 images saved
test: 250/282 images saved
test: 260/282 images saved
test: 270/282 images saved
test: 280/282 images saved
test: 282/282 images saved
✅ test raw predictions comp

,protocol,model,seed,split,mAP50_95,mAP50,mAP75,AP_coral,AP_fish,worst_class_AP,detections
0,time_based,yolo11m,123,validation,0.517411,0.784074,0.540508,0.627403,0.407420,0.407420,8010
1,time_based,yolo11m,123,test,0.454988,0.696380,0.472907,0.533089,0.376887,0.376887,13137


In [14]:
# CELL 10 — CHECKPOINT AND OUTPUT MANIFEST
manifest = {
    "protocol": PROTOCOL,
    "model": MODEL_KEY,
    "seed": SEED,
    "checkpoint_path": str(BEST_CHECKPOINT),
    "checkpoint_sha256": sha256_file(
        BEST_CHECKPOINT
    ),
    "dataset_yaml": str(YAML_PATH),
    "validation_raw_predictions": str(
        RAW_DIR
        / f"validation_{MODEL_KEY}.json"
    ),
    "test_raw_predictions": str(
        RAW_DIR
        / f"test_{MODEL_KEY}.json"
    ),
    "framework_native_results": str(
        RESULTS_DIR
        / "framework_native_results.csv"
    ),
    "common_cocoeval_results": str(
        RESULTS_DIR
        / "common_cocoeval_results.csv"
    ),
}

save_json(
    manifest,
    MANIFEST_DIR / "checkpoint_manifest.json",
)

required_outputs = [
    Path(manifest["checkpoint_path"]),
    Path(manifest["validation_raw_predictions"]),
    Path(manifest["test_raw_predictions"]),
    Path(manifest["framework_native_results"]),
    Path(manifest["common_cocoeval_results"]),
]

missing = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing:
    raise RuntimeError(
        "Missing final outputs:\n"
        + "\n".join(missing)
    )

print("✅ Individual multi-seed run complete.")
print("Run root:", RUN_ROOT)
print("Outputs are isolated by protocol/model/seed.")

✅ Individual multi-seed run complete.
Run root: /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_fullset_dedup_v1/individual/yolo11m/seed_123
Outputs are isolated by protocol/model/seed.


## Output structure

```text
{protocol}/
├── runs/multiseed_fullset_dedup_v1/{model}/seed_{seed}/train/
├── runs_eval/multiseed_fullset_dedup_v1/{model}/seed_{seed}/
└── experiment_logs/multiseed_fullset_dedup_v1/
    └── individual/{model}/seed_{seed}/
        ├── raw_predictions/
        ├── results/
        └── manifests/
```

Jangan memindahkan raw predictions ke folder seed lain. Pairwise WBF akan
membaca file berdasarkan protocol, model, dan seed yang sama.